In [ ]:
# config da camada silver
# centraliza fontes e destinos para que todas as transformações sigam a mesma configuração

try:
    import pyspark.sql.functions as F
    from pyspark.sql import Window
except ModuleNotFoundError as error:
    raise RuntimeError(
        "Este notebook precisa ser executado em um cluster Databricks com PySpark."
    ) from error

# separa claramente a camada preservada da camada em que os dados serão tratados
SCHEMA_ORIGEM = "bronze"
SCHEMA_DESTINO = "silver"

# relaciona cada tabela bronze ao nome em português esperado na silver
TABELAS = {
    "filmes": {"origem": "tb_movies_info", "destino": "tb_info_filmes"},
    "financeiro": {"origem": "tb_movies_financials", "destino": "tb_financeiro_filmes"},
    "metricas": {"origem": "tb_movies_metrics", "destino": "tb_metricas_engajamento"},
    "avaliacoes": {"origem": "tb_movies_reviews", "destino": "tb_avaliacoes_usuarios"},
    "generos": {"origem": "tb_credits_and_tags", "destino": "tb_generos"},
    "pessoas": {"origem": "tb_credits_and_tags", "destino": "tb_pessoas_empresas"},
    "cotacao": {"origem": "tb_cotacao_dolar", "destino": "tb_cotacao_dolar"},
}

# cria o schema sem substituir tabelas existentes antes do processamento
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA_DESTINO}")

In [ ]:
# funções compartilhadas da camada silver
# reúne operações repetidas de leitura, gravação, deduplicação e validação
# essas funções mantêm o mesmo comportamento nas sete tabelas publicadas

# localiza a tabela bronze usando o mapeamento da configuração
def ler_bronze(chave_tabela):
    return spark.table(f"{SCHEMA_ORIGEM}.{TABELAS[chave_tabela]['origem']}")


# relê a tabela persistida para que a validação confira o resultado realmente publicado
def ler_silver(chave_tabela):
    return spark.table(f"{SCHEMA_DESTINO}.{TABELAS[chave_tabela]['destino']}")


# interrompe a execução quando algum campo necessário não existe na origem
def validar_colunas(df, colunas_esperadas, nome_origem):
    ausentes = set(colunas_esperadas).difference(df.columns)
    if ausentes:
        raise ValueError(f"Colunas ausentes em {nome_origem}: {sorted(ausentes)}")


# usa overwrite na silver porque cada execução deve reconstruir a visão tratada atual
def salvar_silver(df, chave_tabela):
    tabela = TABELAS[chave_tabela]["destino"]
    (
        df.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{SCHEMA_DESTINO}.{tabela}")
    )
    print(f"Tabela gravada: {SCHEMA_DESTINO}.{tabela}")
    return ler_silver(chave_tabela)


# garante que a coluna técnica foi traduzida e continua disponível para rastreabilidade
def validar_coluna_ingestao_silver(df, nome_tabela):
    if "ingestion_datetime" in df.columns:
        raise AssertionError(f"A tabela {nome_tabela} ainda possui ingestion_datetime.")
    tipo_ingestao = dict(df.dtypes).get("data_hora_ingestao")
    if tipo_ingestao != "timestamp":
        raise AssertionError(
            f"{nome_tabela}.data_hora_ingestao deve ser TIMESTAMP; encontrado: {tipo_ingestao}."
        )


# compara os tipos publicados com os tipos esperados pela camada gold
def validar_tipos(df, tipos_esperados, nome_tabela):
    tipos_obtidos = {campo.name: campo.dataType.simpleString() for campo in df.schema.fields}
    divergencias = [
        f"{coluna}: esperado {tipo}, obtido {tipos_obtidos.get(coluna)}"
        for coluna, tipo in tipos_esperados.items()
        if tipos_obtidos.get(coluna) != tipo
    ]
    if divergencias:
        raise AssertionError(f"Tipos inválidos em {nome_tabela}: " + "; ".join(divergencias))


# prioriza a carga mais recente, depois a mais completa e por fim um desempate reproduzível
def manter_mais_recente(
    df, chaves, coluna_ingestao="data_hora_ingestao", colunas_completude=None
):
    df_ordenado = df
    ordem = [F.col(coluna_ingestao).desc_nulls_last()]
    colunas_completude = [
        coluna for coluna in (colunas_completude or []) if coluna in df.columns
    ]

    if colunas_completude:
        tipos = dict(df.dtypes)
        completude = F.lit(0)
        for coluna in colunas_completude:
            preenchido = F.col(coluna).isNotNull()
            if tipos.get(coluna) == "string":
                preenchido = preenchido & (F.length(F.trim(F.col(coluna))) > 0)
            completude = completude + F.when(preenchido, F.lit(1)).otherwise(F.lit(0))
        df_ordenado = df_ordenado.withColumn("_completude", completude)
        ordem.append(F.col("_completude").desc())

    colunas_hash = sorted(
        coluna for coluna in df.columns if coluna not in {"_completude", "_desempate_estavel"}
    )
    df_ordenado = df_ordenado.withColumn(
        "_desempate_estavel",
        F.sha2(
            F.concat_ws(
                "||",
                *[F.coalesce(F.col(coluna).cast("string"), F.lit("")) for coluna in colunas_hash],
            ),
            256,
        ),
    )
    ordem.append(F.col("_desempate_estavel").desc())

    janela = Window.partitionBy(*chaves).orderBy(*ordem)
    return (
        df_ordenado.withColumn("_ordem_ingestao", F.row_number().over(janela))
        .where(F.col("_ordem_ingestao") == 1)
        .drop("_ordem_ingestao", "_completude", "_desempate_estavel")
    )


# escolhe a última versão da lista antes do explode para não misturar vínculos antigos e atuais
def manter_ultima_carga_por_chave(df, chaves, coluna_ingestao="data_hora_ingestao"):
    janela = Window.partitionBy(*chaves)
    return (
        df.withColumn("_ultima_ingestao", F.max(coluna_ingestao).over(janela))
        .where(F.col(coluna_ingestao).eqNullSafe(F.col("_ultima_ingestao")))
        .drop("_ultima_ingestao")
    )


# transforma uma regra de qualidade em uma contagem para as validações
def contar_quando(condicao):
    return F.coalesce(
        F.sum(F.when(condicao, F.lit(1)).otherwise(F.lit(0))), F.lit(0)
    ).cast("long")

In [ ]:
# normalizadores da camada silver
# concentra as regras usadas para interpretar datas, números, moedas e nomes sujos
# conversões inválidas viram nulo ou rejeição controlada em vez de interromper o pipeline

# tenta todos os formatos observados para não perder datas válidas escritas de formas diferentes
def converter_data_multiformato(nome_coluna):
    return F.coalesce(
        F.to_date(F.expr(f"try_to_timestamp({nome_coluna}, 'yyyy-MM-dd')")),
        F.to_date(F.expr(f"try_to_timestamp({nome_coluna}, 'dd/MM/yyyy')")),
        F.to_date(F.expr(f"try_to_timestamp({nome_coluna}, 'dd-MM-yyyy')")),
        F.to_date(F.expr(f"try_to_timestamp({nome_coluna}, 'MM-dd-yyyy')")),
        F.to_date(F.expr(f"try_to_timestamp({nome_coluna}, 'yyyy/MM/dd')")),
        F.to_date(F.expr(f"try_to_timestamp({nome_coluna}, 'dd.MM.yyyy')")),
        F.to_date(F.expr(f"try_to_timestamp({nome_coluna}, 'yyyyMMdd')")),
        F.to_date(F.expr(f"try_to_timestamp({nome_coluna})")),
    )


# corrige títulos totalmente em caixa baixa ou alta e preserva grafias que já estão adequadas
def padronizar_titulo(coluna):
    texto = F.trim(F.regexp_replace(coluna.cast("string"), r"\s+", " "))
    titulo_formatado = F.regexp_replace(
        F.initcap(F.regexp_replace(F.lower(texto), "-", "- ")),
        r"-\s+",
        "-",
    )
    return (
        F.when(texto.isNull() | (texto == ""), F.lit(None).cast("string"))
        .when((texto == F.lower(texto)) | (texto == F.upper(texto)), titulo_formatado)
        .otherwise(texto)
    )


# cria uma chave auxiliar de obra sem substituir o id natural recebido da origem
def criar_chave_obra_canonica(id_imdb, titulo, data_lancamento, id_filme):
    imdb_normalizado = F.lower(F.trim(id_imdb.cast("string")))
    titulo_normalizado = F.lower(
        F.regexp_replace(F.trim(titulo.cast("string")), r"[^\p{L}\p{N}]+", "")
    )
    return (
        F.when(
            imdb_normalizado.rlike(r"^tt[0-9]+$"),
            F.concat(F.lit("imdb:"), imdb_normalizado),
        )
        .when(
            (F.length(titulo_normalizado) > 0) & data_lancamento.isNotNull(),
            F.concat(
                F.lit("titulo:"),
                F.sha2(
                    F.concat_ws("|", titulo_normalizado, data_lancamento.cast("string")),
                    256,
                ),
            ),
        )
        .otherwise(F.concat(F.lit("origem:"), id_filme.cast("string")))
    )


# reúne textos que representam ausência de informação
MARCADORES_AUSENCIA = r"^(|unknown|não informado|sem informação|sem informacao|nenhum|none|na|n/a|null)$"


# remove símbolos de moeda e interpreta ponto ou vírgula conforme a posição no valor
def normalizar_valor_monetario(coluna):
    texto = F.lower(F.trim(coluna.cast("string")))
    valor_sem_simbolos = F.regexp_replace(texto, r"[^0-9,.\-]", "")
    valor_sem_simbolos = F.when(
        texto.rlike(r"^\s*\(.*\)\s*$"), F.concat(F.lit("-"), valor_sem_simbolos)
    ).otherwise(valor_sem_simbolos)

    possui_virgula = F.instr(valor_sem_simbolos, ",") > 0
    possui_ponto = F.instr(valor_sem_simbolos, ".") > 0
    ultimo_separador = F.regexp_extract(valor_sem_simbolos, r"([.,])[0-9]+$", 1)
    virgula_como_milhar = valor_sem_simbolos.rlike(r"^-?[0-9]{1,3}(,[0-9]{3})+$")
    ponto_como_milhar = valor_sem_simbolos.rlike(r"^-?[0-9]{1,3}(\.[0-9]{3})+$")

    valor_normalizado = (
        F.when(
            possui_virgula & possui_ponto & (ultimo_separador == ","),
            F.regexp_replace(F.regexp_replace(valor_sem_simbolos, r"\.", ""), ",", "."),
        )
        .when(
            possui_virgula & possui_ponto & (ultimo_separador == "."),
            F.regexp_replace(valor_sem_simbolos, ",", ""),
        )
        .when(
            possui_virgula & ~possui_ponto & virgula_como_milhar,
            F.regexp_replace(valor_sem_simbolos, ",", ""),
        )
        .when(
            possui_virgula & ~possui_ponto,
            F.regexp_replace(valor_sem_simbolos, ",", "."),
        )
        .when(
            possui_ponto & ~possui_virgula & ponto_como_milhar,
            F.regexp_replace(valor_sem_simbolos, r"\.", ""),
        )
        .otherwise(valor_sem_simbolos)
    )

    return F.when(
        texto.isNull() | texto.rlike(MARCADORES_AUSENCIA) | (valor_sem_simbolos == ""),
        F.lit(None).cast("string"),
    ).otherwise(valor_normalizado)


# converte abreviações como 1.5m sem perder a escala do valor
def multiplicador_sufixo_monetario(coluna):
    sufixo = F.upper(F.regexp_extract(F.trim(coluna.cast("string")), r"([KMB])\s*$", 1))
    return (
        F.when(sufixo == "K", F.lit(1_000))
        .when(sufixo == "M", F.lit(1_000_000))
        .when(sufixo == "B", F.lit(1_000_000_000))
        .otherwise(F.lit(1))
        .cast("long")
    )


# converte separadores decimais sem aceitar textos deslocados como se fossem métricas
def normalizar_numero_decimal(coluna):
    texto = F.lower(F.trim(coluna.cast("string")))
    valor = F.regexp_replace(texto, r"\s+", "")
    formato_numerico = valor.rlike(r"^-?[0-9]+([.,][0-9]+)*$")
    quantidade_virgulas = F.length(valor) - F.length(F.regexp_replace(valor, ",", ""))
    quantidade_pontos = F.length(valor) - F.length(F.regexp_replace(valor, r"\.", ""))
    possui_virgula = quantidade_virgulas > 0
    possui_ponto = quantidade_pontos > 0
    ultimo_separador = F.regexp_extract(valor, r"([.,])[0-9]+$", 1)

    valor_normalizado = (
        F.when(
            possui_virgula & possui_ponto & (ultimo_separador == ","),
            F.regexp_replace(F.regexp_replace(valor, r"\.", ""), ",", "."),
        )
        .when(
            possui_virgula & possui_ponto & (ultimo_separador == "."),
            F.regexp_replace(valor, ",", ""),
        )
        .when(
            possui_virgula & ~possui_ponto & (quantidade_virgulas == 1),
            F.regexp_replace(valor, ",", "."),
        )
        .when(possui_virgula & ~possui_ponto, F.regexp_replace(valor, ",", ""))
        .when(
            possui_ponto & ~possui_virgula & (quantidade_pontos > 1),
            F.regexp_replace(valor, r"\.", ""),
        )
        .otherwise(valor)
    )
    return F.when(
        texto.isNull() | ~formato_numerico, F.lit(None).cast("string")
    ).otherwise(valor_normalizado)


# aceita contagens inteiras e rejeita valores fracionários ou textuais
def normalizar_numero_inteiro(coluna):
    texto = F.lower(F.trim(coluna.cast("string")))
    valor = F.regexp_replace(texto, r"\s+", "")
    inteiro_simples = valor.rlike(r"^-?[0-9]+$")
    inteiro_com_milhar = valor.rlike(r"^-?[0-9]{1,3}([.,][0-9]{3})+$")
    inteiro_com_decimal_zero = valor.rlike(r"^-?[0-9]+[.,]0+$")
    return (
        F.when(texto.isNull(), F.lit(None).cast("string"))
        .when(inteiro_simples, valor)
        .when(inteiro_com_milhar, F.regexp_replace(valor, r"[.,]", ""))
        .when(inteiro_com_decimal_zero, F.regexp_replace(valor, r"[.,]0+$", ""))
        .otherwise(F.lit(None).cast("string"))
    )


# define os valores que não representam uma pessoa ou empresa válida
MARCADORES_ENTIDADE_AUSENTE = [
    "", "-", "--", "n/a", "na", "null", "none", "nenhum",
    "unknown", "não informado", "sem informação", "sem informacao", "[]",
]

# reúne valores de outros domínios que aparecem isolados nas listas por causa do column shift
RESIDUOS_ENTIDADE_CONTEXTO = [
    "action", "adventure", "animation", "comedy", "crime", "documentary",
    "drama", "family", "fantasy", "history", "horror", "music",
    "mystery", "romance", "science fiction", "tv movie", "thriller",
    "war", "western", "english", "italian", "french", "spanish",
    "german", "portuguese", "japanese", "korean", "mandarin",
    "cantonese", "arabic", "hindi", "russian", "turkish", "polish",
    "dutch", "swedish", "danish", "norwegian", "finnish", "greek",
    "hebrew", "thai", "indonesian", "vietnamese", "united states",
    "united kingdom", "france", "germany", "italy", "spain", "canada",
    "brazil", "japan", "china", "south korea", "india", "australia",
]


# ajusta a capitalização sem alterar nomes que já possuem grafia mista
def padronizar_nome_entidade(coluna):
    texto = F.trim(F.regexp_replace(coluna, r"\s+", " "))
    tokens = F.split(texto, " ")
    tokens_minusculos = F.transform(
        tokens,
        lambda token: F.when(F.instr(token, ".") > 0, F.upper(token)).otherwise(F.initcap(token)),
    )
    tokens_maiusculos = F.transform(
        tokens,
        lambda token: F.when(
            (F.length(token) <= 3) | (F.instr(token, ".") > 0), token
        ).otherwise(F.initcap(F.lower(token))),
    )
    return (
        F.when(texto == F.lower(texto), F.concat_ws(" ", tokens_minusculos))
        .when(texto == F.upper(texto), F.concat_ws(" ", tokens_maiusculos))
        .otherwise(texto)
    )


# classifica resíduos deslocados para descartá-los sem perder a explicação do motivo
def classificar_residuo_entidade(coluna):
    texto = F.lower(F.trim(coluna))
    return (
        F.when(texto.isNull() | texto.isin(MARCADORES_ENTIDADE_AUSENTE), F.lit("marcador de ausência"))
        .when(texto.isin(RESIDUOS_ENTIDADE_CONTEXTO), F.lit("valor de outro domínio"))
        .when(texto.rlike(r"^[+-]?[0-9]+([.,][0-9]+)*$"), F.lit("valor numérico deslocado"))
        .when(texto.rlike(r"^[0-9]{4}-[0-9]{1,2}-[0-9]{1,2}([ t].*)?$"), F.lit("data deslocada"))
        .when(texto.rlike(r"^[0-9]{1,2}[/-][0-9]{1,2}[/-][0-9]{2,4}([ t].*)?$"), F.lit("data deslocada"))
        .when(~texto.rlike(r".*\p{L}.*"), F.lit("texto sem letras"))
        .when(texto.rlike(r"^(https?://|www\.).*") | texto.rlike(r"^.*@.*\..*$"), F.lit("link ou contato deslocado"))
        .when(
            texto.rlike(
                r"^(released|post production|in production|planned|rumored|canceled|cancelled|lançado|pós-produção|em produção|planejado|rumores|cancelado|true|false)$"
            ),
            F.lit("atributo contextual deslocado"),
        )
        .when(F.length(texto) > 150, F.lit("texto descritivo extenso"))
        .otherwise(F.lit(None).cast("string"))
    )

In [ ]:
# tabela silver.tb_info_filmes
# transforma os metadados dos filmes sem alterar as tabelas preservadas na bronze
# mantém o id natural da origem e cria uma chave auxiliar somente para análises por obra

# carrega a tabela de filmes da bronze e confere os campos necessários
df_bronze = ler_bronze("filmes")
validar_colunas(
    df_bronze,
    {
        "id", "tconst", "title", "original_title", "original_language",
        "release_date", "runtime", "status", "overview", "tagline", "ingestion_datetime",
    },
    "bronze.tb_movies_info",
)

# remove diferenças de caixa, espaço e hífen antes de traduzir os status conhecidos
status_normalizado = F.lower(
    F.trim(F.regexp_replace(F.coalesce(F.col("status"), F.lit("")), r"[\s_-]+", " "))
)
status_traduzido = (
    F.when(status_normalizado == "released", F.lit("Lançado"))
    .when(status_normalizado == "post production", F.lit("Pós-Produção"))
    .when(status_normalizado == "in production", F.lit("Em Produção"))
    .when(status_normalizado == "planned", F.lit("Planejado"))
    .when(status_normalizado == "rumored", F.lit("Rumores"))
    .when(status_normalizado.isin("canceled", "cancelled"), F.lit("Cancelado"))
    .otherwise(F.lit("Não Informado"))
)

# aplica os nomes em português, converte os tipos e deriva o ano da data já tratada
df_tratado = (
    df_bronze
    .withColumn("data_hora_ingestao", F.col("ingestion_datetime"))
    .withColumn("id_filme", F.trim(F.col("id").cast("string")))
    # tconst é o identificador externo do IMDb e é preservado como id_imdb para integrações futuras
    .withColumn("id_imdb", F.lower(F.trim(F.col("tconst"))))
    .withColumn("titulo", padronizar_titulo(F.col("title")))
    .withColumn("titulo_original", padronizar_titulo(F.col("original_title")))
    .withColumn("idioma_original", F.trim(F.col("original_language")))
    .withColumn("duracao_minutos", F.expr("try_cast(runtime AS INT)"))
    .withColumn("status_filme", status_traduzido)
    .withColumn("sinopse", F.trim(F.col("overview")))
    .withColumn("frase_divulgacao", F.trim(F.col("tagline")))
    .withColumn("data_texto", F.trim(F.col("release_date")))
    .withColumn("data_lancamento", converter_data_multiformato("data_texto"))
    .withColumn("ano_lancamento", F.year("data_lancamento").cast("INT"))
)

df_datas_nao_convertidas = (
    df_tratado
    .where(F.col("data_texto").isNotNull() & (F.col("data_texto") != "") & F.col("data_lancamento").isNull())
    .select("id_filme", F.col("release_date").alias("data_origem"), "data_texto")
    .dropDuplicates(["id_filme", "data_texto"])
)

# escolhe a versão mais recente e completa de cada id natural recebido da bronze
# o id natural é usado porque identifica o mesmo filme entre diferentes cargas da origem
df_filmes_por_id = (
    manter_mais_recente(
        df_tratado.where(F.col("id_filme").isNotNull() & (F.col("id_filme") != "")),
        ["id_filme"],
        colunas_completude=[
            "titulo", "titulo_original", "data_lancamento", "duracao_minutos",
            "idioma_original", "sinopse", "frase_divulgacao",
        ],
    )
    # a chave canônica reduz a dependência de um único identificador da origem
    .withColumn(
        "id_obra_canonica",
        criar_chave_obra_canonica(
            F.col("id_imdb"), F.col("titulo"),
            F.col("data_lancamento"), F.col("id_filme"),
        ),
    )
)

# deixa explícito o vínculo entre o registro da origem e a obra identificada como equivalente
df_mapa_filmes = df_filmes_por_id.select("id_filme", "id_obra_canonica").dropDuplicates()

# publica uma linha por id natural conforme o mapeamento exigido no enunciado
df_info_silver = (
    df_filmes_por_id
    .select(
        "id_filme", "id_obra_canonica", "id_imdb", "titulo", "titulo_original", "idioma_original",
        "data_lancamento", "ano_lancamento", "duracao_minutos", "status_filme",
        "sinopse", "frase_divulgacao", "data_hora_ingestao",
    )
)

df_info_validacao = salvar_silver(df_info_silver, "filmes")

In [ ]:
# validação de silver.tb_info_filmes
# impede ids naturais repetidos, tipos incorretos ou anos incompatíveis com a data

validar_coluna_ingestao_silver(df_info_validacao, TABELAS["filmes"]["destino"])
validar_tipos(
    df_info_validacao,
    {
        "id_filme": "string", "id_obra_canonica": "string",
        "data_lancamento": "date", "ano_lancamento": "int",
        "duracao_minutos": "int", "data_hora_ingestao": "timestamp",
    },
    TABELAS["filmes"]["destino"],
)

# resume unicidade, datas e anos em uma única validação
resumo_info = df_info_validacao.agg(
    F.count("*").alias("total"),
    F.countDistinct("id_filme").alias("ids_distintos"),
    contar_quando(F.col("id_filme").isNull() | (F.trim("id_filme") == "")).alias("ids_invalidos"),
    contar_quando(F.col("data_lancamento").isNull()).alias("datas_nulas"),
    contar_quando(
        F.col("data_lancamento").isNotNull()
        & (F.col("ano_lancamento") != F.year("data_lancamento"))
    ).alias("anos_inconsistentes"),
    contar_quando(~F.col("status_filme").isin(
        "Lançado", "Pós-Produção", "Em Produção", "Planejado",
        "Rumores", "Cancelado", "Não Informado",
    )).alias("status_invalidos"),
).first()

df_ids_agrupados = (
    df_info_validacao
    .groupBy("id_obra_canonica")
    .agg(
        F.countDistinct("id_filme").alias("quantidade_ids_origem"),
        F.sort_array(F.collect_set("titulo")).alias("titulos_encontrados"),
    )
    .where(F.col("quantidade_ids_origem") > 1)
)
mapeamentos_ambiguos = (
    df_mapa_filmes
    .groupBy("id_filme")
    .agg(F.countDistinct("id_obra_canonica").alias("quantidade_destinos"))
    .where(F.col("quantidade_destinos") > 1)
    .count()
)
chaves_de_obra_invalidas = df_info_validacao.where(
    F.col("id_obra_canonica").isNull()
    | ~F.col("id_obra_canonica").rlike(r"^(imdb:tt[0-9]+|titulo:[0-9a-f]{64}|origem:.+)$")
).count()

if resumo_info["total"] != resumo_info["ids_distintos"]:
    raise AssertionError("Há ids naturais de filme duplicados na Silver.")
if resumo_info["ids_invalidos"] != 0:
    raise AssertionError("Há ids naturais de filme vazios na Silver.")
if mapeamentos_ambiguos != 0:
    raise AssertionError("Um id natural aponta para mais de uma obra canônica.")
if chaves_de_obra_invalidas != 0:
    raise AssertionError("Há chaves auxiliares de obra ausentes ou inválidas.")
if resumo_info["anos_inconsistentes"] != 0:
    raise AssertionError("Há anos de lançamento inconsistentes com a data.")
if resumo_info["status_invalidos"] != 0:
    raise AssertionError("Há status fora do domínio traduzido na Silver.")

display(df_datas_nao_convertidas.orderBy("data_texto").limit(20))
display(df_ids_agrupados.orderBy(F.col("quantidade_ids_origem").desc()).limit(20))
display(df_info_validacao.groupBy("status_filme").count().orderBy(F.col("count").desc()))
print(
    f"Filmes: {resumo_info['total']} | Datas nulas: {resumo_info['datas_nulas']} | "
    f"Datas de origem não convertidas: {df_datas_nao_convertidas.count()}"
)

In [ ]:
# tabela silver.tb_financeiro_filmes
# interpreta orçamento e receita sem transformar textos inválidos em valores financeiros

df_financeiro_bronze = ler_bronze("financeiro")
validar_colunas(
    df_financeiro_bronze,
    {"id", "budget", "revenue", "ingestion_datetime"},
    "bronze.tb_movies_financials",
)

# remove moedas e separadores antes do decimal para evitar conversões silenciosamente erradas
df_financeiro = (
    df_financeiro_bronze
    .withColumn("data_hora_ingestao", F.col("ingestion_datetime"))
    .withColumn("id_filme", F.trim(F.col("id").cast("string")))
    .withColumn("orcamento_normalizado", normalizar_valor_monetario(F.col("budget")))
    .withColumn("receita_normalizada", normalizar_valor_monetario(F.col("revenue")))
    .withColumn("_multiplicador_orcamento", multiplicador_sufixo_monetario(F.col("budget")))
    .withColumn("_multiplicador_receita", multiplicador_sufixo_monetario(F.col("revenue")))
    .withColumn("_orcamento_base", F.expr("try_cast(orcamento_normalizado AS DECIMAL(20,4))"))
    .withColumn("_receita_base", F.expr("try_cast(receita_normalizada AS DECIMAL(20,4))"))
    .withColumn(
        "orcamento_usd",
        F.expr("try_cast(_orcamento_base * _multiplicador_orcamento AS DECIMAL(18,2))"),
    )
    .withColumn(
        "receita_usd",
        F.expr("try_cast(_receita_base * _multiplicador_receita AS DECIMAL(18,2))"),
    )
    .withColumn("orcamento_usd", F.when(F.col("orcamento_usd") > 0, F.col("orcamento_usd")))
    .withColumn("receita_usd", F.when(F.col("receita_usd") > 0, F.col("receita_usd")))
    .where(F.col("id_filme").isNotNull() & (F.col("id_filme") != ""))
)
df_financeiro = manter_mais_recente(
    df_financeiro,
    ["id_filme"],
    colunas_completude=["orcamento_usd", "receita_usd"],
)
# mantém o mesmo id natural usado na tabela de informações para preservar a rastreabilidade

# usa a cotação mais recente porque a base não informa a data de cada transação financeira
df_cotacao_atual = (
    ler_bronze("cotacao")
    .withColumn("data_hora_cotacao", F.to_timestamp("dataHoraCotacao"))
    .where(F.col("cotacaoCompra") > 0)
    .orderBy(F.col("data_hora_cotacao").desc())
    .limit(1)
    .select(F.col("cotacaoCompra").cast("DECIMAL(12,6)").alias("cotacao_dolar_brl"))
)
if df_cotacao_atual.isEmpty():
    raise ValueError("A Bronze não possui uma cotação PTAX válida para a conversão financeira.")

# converte os valores para reais e deriva lucro e margem sem dividir por zero
df_financeiro_silver = (
    df_financeiro
    .crossJoin(df_cotacao_atual)
    .withColumn("orcamento_brl", (F.col("orcamento_usd") * F.col("cotacao_dolar_brl")).cast("DECIMAL(18,2)"))
    .withColumn("receita_brl", (F.col("receita_usd") * F.col("cotacao_dolar_brl")).cast("DECIMAL(18,2)"))
    .withColumn("lucro_usd", (F.col("receita_usd") - F.col("orcamento_usd")).cast("DECIMAL(18,2)"))
    .withColumn("lucro_brl", (F.col("receita_brl") - F.col("orcamento_brl")).cast("DECIMAL(18,2)"))
    .withColumn(
        "margem_lucro_percentual",
        F.when(
            F.col("receita_usd") > 0,
            (F.col("lucro_usd") / F.col("receita_usd") * 100).cast("DECIMAL(10,2)"),
        ),
    )
    .select(
        "id_filme", "orcamento_usd", "receita_usd", "cotacao_dolar_brl",
        "orcamento_brl", "receita_brl", "lucro_usd", "lucro_brl",
        "margem_lucro_percentual", "data_hora_ingestao",
    )
)

df_financeiro_validacao = salvar_silver(df_financeiro_silver, "financeiro")

In [ ]:
# validação de silver.tb_financeiro_filmes
# garante um registro por id natural e rejeita valores financeiros zerados ou negativos

validar_coluna_ingestao_silver(df_financeiro_validacao, TABELAS["financeiro"]["destino"])
validar_tipos(
    df_financeiro_validacao,
    {
        "id_filme": "string", "orcamento_usd": "decimal(18,2)",
        "receita_usd": "decimal(18,2)", "cotacao_dolar_brl": "decimal(12,6)",
        "orcamento_brl": "decimal(18,2)", "receita_brl": "decimal(18,2)",
        "lucro_usd": "decimal(18,2)", "lucro_brl": "decimal(18,2)",
        "margem_lucro_percentual": "decimal(10,2)", "data_hora_ingestao": "timestamp",
    },
    TABELAS["financeiro"]["destino"],
)

# confere unicidade, preenchimento e valores financeiros permitidos
resumo_financeiro = df_financeiro_validacao.agg(
    F.count("*").alias("total"),
    F.countDistinct("id_filme").alias("ids_distintos"),
    contar_quando(F.col("id_filme").isNull() | (F.trim("id_filme") == "")).alias("ids_invalidos"),
    F.count("orcamento_usd").alias("com_orcamento"),
    F.count("receita_usd").alias("com_receita"),
    contar_quando(
        (F.col("orcamento_usd").isNotNull() & (F.col("orcamento_usd") <= 0))
        | (F.col("receita_usd").isNotNull() & (F.col("receita_usd") <= 0))
    ).alias("valores_nao_positivos"),
    contar_quando(F.col("cotacao_dolar_brl").isNull() | (F.col("cotacao_dolar_brl") <= 0)).alias("cotacoes_invalidas"),
    contar_quando(
        ~F.col("orcamento_brl").eqNullSafe(
            (F.col("orcamento_usd") * F.col("cotacao_dolar_brl")).cast("DECIMAL(18,2)")
        )
        | ~F.col("receita_brl").eqNullSafe(
            (F.col("receita_usd") * F.col("cotacao_dolar_brl")).cast("DECIMAL(18,2)")
        )
        | ~F.col("lucro_usd").eqNullSafe(
            (F.col("receita_usd") - F.col("orcamento_usd")).cast("DECIMAL(18,2)")
        )
        | ~F.col("lucro_brl").eqNullSafe(
            (F.col("receita_brl") - F.col("orcamento_brl")).cast("DECIMAL(18,2)")
        )
        | ~F.col("margem_lucro_percentual").eqNullSafe(
            F.when(
                F.col("receita_usd") > 0,
                (F.col("lucro_usd") / F.col("receita_usd") * 100).cast("DECIMAL(10,2)"),
            )
        )
    ).alias("calculos_inconsistentes"),
).first()

if resumo_financeiro["total"] != resumo_financeiro["ids_distintos"]:
    raise AssertionError("Há ids duplicados na Silver financeira.")
if resumo_financeiro["ids_invalidos"] != 0:
    raise AssertionError("Há ids naturais vazios na Silver financeira.")
if resumo_financeiro["valores_nao_positivos"] != 0:
    raise AssertionError("Há valores financeiros não positivos na Silver.")
if resumo_financeiro["cotacoes_invalidas"] != 0 or resumo_financeiro["calculos_inconsistentes"] != 0:
    raise AssertionError("Há cotações inválidas ou cálculos financeiros inconsistentes na Silver.")

display(
    df_financeiro_validacao.select(
        "id_filme", "orcamento_usd", "receita_usd", "lucro_usd", "margem_lucro_percentual"
    ).limit(10)
)
print(
    f"Filmes: {resumo_financeiro['total']} | Orçamentos válidos: {resumo_financeiro['com_orcamento']} | "
    f"Receitas válidas: {resumo_financeiro['com_receita']}"
)

In [ ]:
# tabela silver.tb_metricas_engajamento
# converte popularidade, notas e votos sem aceitar textos deslocados como números
# invalida escalas impossíveis e preserva como nulo o que não pode ser interpretado

df_metricas_bronze = ler_bronze("metricas")
validar_colunas(
    df_metricas_bronze,
    {"id", "popularity", "vote_average", "vote_count", "averageRating", "numVotes", "ingestion_datetime"},
    "bronze.tb_movies_metrics",
)

# normaliza separadores e tipos antes de aplicar as faixas permitidas pelo enunciado
df_metricas = (
    df_metricas_bronze
    .withColumn("data_hora_ingestao", F.col("ingestion_datetime"))
    .withColumn("id_filme", F.trim(F.col("id").cast("string")))
    .withColumn("popularidade_texto", F.trim("popularity"))
    .withColumn("nota_media_tmdb_texto", F.trim("vote_average"))
    .withColumn("qtd_votos_tmdb_texto", F.trim("vote_count"))
    .withColumn("nota_media_imdb_texto", F.trim("averageRating"))
    .withColumn("qtd_votos_imdb_texto", F.trim("numVotes"))
    .withColumn("popularidade_normalizada", normalizar_numero_decimal(F.col("popularidade_texto")))
    .withColumn("nota_media_tmdb_normalizada", normalizar_numero_decimal(F.col("nota_media_tmdb_texto")))
    .withColumn("qtd_votos_tmdb_normalizada", normalizar_numero_inteiro(F.col("qtd_votos_tmdb_texto")))
    .withColumn("nota_media_imdb_normalizada", normalizar_numero_decimal(F.col("nota_media_imdb_texto")))
    .withColumn("qtd_votos_imdb_normalizada", normalizar_numero_inteiro(F.col("qtd_votos_imdb_texto")))
    .withColumn("popularidade", F.expr("try_cast(popularidade_normalizada AS DOUBLE)"))
    .withColumn("nota_media_tmdb", F.expr("try_cast(nota_media_tmdb_normalizada AS DOUBLE)"))
    .withColumn("qtd_votos_tmdb", F.expr("try_cast(qtd_votos_tmdb_normalizada AS INT)"))
    .withColumn("nota_media_imdb", F.expr("try_cast(nota_media_imdb_normalizada AS DOUBLE)"))
    .withColumn("qtd_votos_imdb", F.expr("try_cast(qtd_votos_imdb_normalizada AS INT)"))
    .withColumn("popularidade", F.when(F.col("popularidade") >= 0, F.col("popularidade")))
    .withColumn("qtd_votos_tmdb", F.when(F.col("qtd_votos_tmdb") >= 0, F.col("qtd_votos_tmdb")))
    .withColumn("qtd_votos_imdb", F.when(F.col("qtd_votos_imdb") >= 0, F.col("qtd_votos_imdb")))
    .withColumn("nota_media_tmdb", F.when(F.col("nota_media_tmdb").between(0, 10), F.col("nota_media_tmdb")))
    .withColumn("nota_media_imdb", F.when(F.col("nota_media_imdb").between(0, 10), F.col("nota_media_imdb")))
)

# escolhe a carga mais recente e completa de cada id natural
df_metricas = manter_mais_recente(
    df_metricas.where(
        F.col("id_filme").isNotNull() & (F.col("id_filme") != "")
    ),
    ["id_filme"],
    colunas_completude=[
        "popularidade", "nota_media_tmdb", "qtd_votos_tmdb",
        "nota_media_imdb", "qtd_votos_imdb",
    ],
)

# procura números com formato de ano acompanhados por texto nas demais colunas numéricas
# a combinação dos dois sinais evita remover uma popularidade alta apenas pelo seu valor
texto_deslocado_metricas = (
    (
        F.col("nota_media_tmdb_texto").rlike(r".*\p{L}.*")
        & F.col("nota_media_tmdb_normalizada").isNull()
    )
    | (
        F.col("qtd_votos_tmdb_texto").rlike(r".*\p{L}.*")
        & F.col("qtd_votos_tmdb_normalizada").isNull()
    )
    | (
        F.col("nota_media_imdb_texto").rlike(r".*\p{L}.*")
        & F.col("nota_media_imdb_normalizada").isNull()
    )
    | (
        F.col("qtd_votos_imdb_texto").rlike(r".*\p{L}.*")
        & F.col("qtd_votos_imdb_normalizada").isNull()
    )
)
popularidade_com_formato_ano = (
    F.col("popularidade").between(1888, F.year(F.current_date()) + F.lit(5))
    & (F.col("popularidade") == F.floor(F.col("popularidade")))
)
df_metricas = df_metricas.withColumn(
    "_popularidade_deslocada", popularidade_com_formato_ano & texto_deslocado_metricas
)

# mantém uma amostra auditável antes de anular somente os casos com evidência de column shift
df_popularidades_deslocadas = (
    df_metricas.where(F.col("_popularidade_deslocada"))
    .select(
        "id_filme", "popularidade_texto", "nota_media_tmdb_texto",
        "qtd_votos_tmdb_texto", "nota_media_imdb_texto", "qtd_votos_imdb_texto",
    )
    .join(ler_silver("filmes").select("id_filme", "titulo"), "id_filme", "left")
)
df_metricas_tratadas = df_metricas.withColumn(
    "popularidade",
    F.when(F.col("_popularidade_deslocada"), F.lit(None).cast("double"))
    .otherwise(F.col("popularidade")),
)

# publica uma linha por id natural depois da limpeza e da deduplicação
df_metricas_silver = (
    df_metricas_tratadas
    .select(
        "id_filme", "popularidade", "nota_media_tmdb", "qtd_votos_tmdb",
        "nota_media_imdb", "qtd_votos_imdb", "data_hora_ingestao",
    )
)
df_metricas_validacao = salvar_silver(df_metricas_silver, "metricas")

In [ ]:
# validação de silver.tb_metricas_engajamento
# confirma que notas permanecem entre zero e dez e que contagens não ficaram negativas

validar_coluna_ingestao_silver(df_metricas_validacao, TABELAS["metricas"]["destino"])
validar_tipos(
    df_metricas_validacao,
    {
        "id_filme": "string", "popularidade": "double", "nota_media_tmdb": "double",
        "qtd_votos_tmdb": "int", "nota_media_imdb": "double", "qtd_votos_imdb": "int",
        "data_hora_ingestao": "timestamp",
    },
    TABELAS["metricas"]["destino"],
)

# confere unicidade, notas entre zero e dez e métricas não negativas
resumo_metricas = df_metricas_validacao.agg(
    F.count("*").alias("total"),
    F.countDistinct("id_filme").alias("ids_distintos"),
    contar_quando(F.col("id_filme").isNull() | (F.trim("id_filme") == "")).alias("ids_invalidos"),
    contar_quando(
        (F.col("nota_media_tmdb").isNotNull() & ~F.col("nota_media_tmdb").between(0, 10))
        | (F.col("nota_media_imdb").isNotNull() & ~F.col("nota_media_imdb").between(0, 10))
    ).alias("notas_invalidas"),
    contar_quando(
        (F.col("popularidade").isNotNull() & (F.col("popularidade") < 0))
        | (F.col("qtd_votos_tmdb").isNotNull() & (F.col("qtd_votos_tmdb") < 0))
        | (F.col("qtd_votos_imdb").isNotNull() & (F.col("qtd_votos_imdb") < 0))
    ).alias("metricas_negativas"),
).first()

popularidades_suspeitas_restantes = df_metricas_tratadas.where(
    F.col("_popularidade_deslocada") & F.col("popularidade").isNotNull()
).count()

if resumo_metricas["total"] != resumo_metricas["ids_distintos"]:
    raise AssertionError("Há ids duplicados na Silver de métricas.")
if resumo_metricas["ids_invalidos"] != 0:
    raise AssertionError("Há ids naturais vazios na Silver de métricas.")
if resumo_metricas["notas_invalidas"] != 0 or resumo_metricas["metricas_negativas"] != 0:
    raise AssertionError("Há notas fora da escala ou métricas negativas na Silver.")
if popularidades_suspeitas_restantes != 0:
    raise AssertionError("Há popularidades com sinais de column shift que não foram anuladas.")

display(df_popularidades_deslocadas.orderBy("popularidade_texto", "titulo").limit(20))
display(df_metricas_validacao.limit(10))
print(
    f"Filmes: {resumo_metricas['total']} | Popularidades removidas por column shift: "
    f"{df_popularidades_deslocadas.count()}"
)

In [ ]:
# tabela silver.tb_avaliacoes_usuarios
# preserva uma avaliação por combinação de filme, usuário, nota e comentário
# aplica a escala de zero a dez e explicita quando o comentário não foi informado

# carrega as avaliações individuais e confere os campos esperados
df_avaliacoes_bronze = ler_bronze("avaliacoes")
validar_colunas(
    df_avaliacoes_bronze,
    {"id", "nome", "nota", "comentario", "ingestion_datetime"},
    "bronze.tb_movies_reviews",
)

df_avaliacoes_origem = (
    df_avaliacoes_bronze
    .withColumn("data_hora_ingestao", F.col("ingestion_datetime"))
    .withColumn("id_filme", F.trim(F.col("id").cast("string")))
)
# deduplica pela combinação integral solicitada sem consolidar ids diferentes
chaves_avaliacao_origem = ["id_filme", "nome", "nota", "comentario"]
df_avaliacoes_sem_duplicatas = manter_mais_recente(
    df_avaliacoes_origem, chaves_avaliacao_origem
)
id_avaliacao_valido = F.col("id_filme").isNotNull() & (F.col("id_filme") != "")
df_avaliacoes_sem_id = df_avaliacoes_sem_duplicatas.where(~id_avaliacao_valido)

# trata nota e comentário depois que a unicidade original já foi preservada
df_avaliacoes_silver = (
    df_avaliacoes_sem_duplicatas
    .where(id_avaliacao_valido)
    .withColumn("nome_usuario", F.trim("nome"))
    .withColumn("nota_usuario_normalizada", normalizar_numero_decimal(F.col("nota")))
    .withColumn("nota_usuario", F.expr("try_cast(nota_usuario_normalizada AS DOUBLE)"))
    .withColumn("nota_usuario", F.when(F.col("nota_usuario").between(0, 10), F.col("nota_usuario")))
    .withColumn("comentario_usuario", F.trim("comentario"))
    .withColumn(
        "comentario_usuario",
        F.when(
            F.col("comentario_usuario").isNull() | (F.col("comentario_usuario") == ""),
            F.lit("Sem comentário"),
        ).otherwise(F.col("comentario_usuario")),
    )
    .select("id_filme", "nome_usuario", "nota_usuario", "comentario_usuario", "data_hora_ingestao")
)

df_avaliacoes_validacao = salvar_silver(df_avaliacoes_silver, "avaliacoes")

In [ ]:
# validação de silver.tb_avaliacoes_usuarios
# confere duplicatas, notas e comentários após o tratamento

validar_coluna_ingestao_silver(df_avaliacoes_validacao, TABELAS["avaliacoes"]["destino"])
validar_tipos(
    df_avaliacoes_validacao,
    {"id_filme": "string", "nota_usuario": "double", "data_hora_ingestao": "timestamp"},
    TABELAS["avaliacoes"]["destino"],
)

# compara entrada e saída para garantir que somente duplicatas integrais foram removidas
duplicados_origem = (
    df_avaliacoes_sem_duplicatas
    .groupBy(*chaves_avaliacao_origem).count()
    .where(F.col("count") > 1)
    .count()
)
resumo_avaliacoes = df_avaliacoes_validacao.agg(
    F.count("*").alias("total"),
    contar_quando(F.col("id_filme").isNull() | (F.trim("id_filme") == "")).alias("ids_invalidos"),
    contar_quando(F.col("nota_usuario").isNotNull() & ~F.col("nota_usuario").between(0, 10)).alias("notas_invalidas"),
    contar_quando(F.col("comentario_usuario").isNull() | (F.trim("comentario_usuario") == "")).alias("comentarios_vazios"),
    contar_quando(F.col("comentario_usuario") == "Sem comentário").alias("comentarios_padronizados"),
).first()
quantidade_esperada = df_avaliacoes_sem_duplicatas.where(id_avaliacao_valido).count()
duplicatas_removidas = df_avaliacoes_origem.count() - df_avaliacoes_sem_duplicatas.count()

if duplicados_origem != 0 or resumo_avaliacoes["total"] != quantidade_esperada:
    raise AssertionError("A deduplicação integral das avaliações ficou inconsistente.")
if any(resumo_avaliacoes[c] != 0 for c in ["ids_invalidos", "notas_invalidas", "comentarios_vazios"]):
    raise AssertionError("Há ids, notas ou comentários inválidos na Silver de avaliações.")

display(df_avaliacoes_validacao.limit(10))
print(
    f"Avaliações sem id_filme: {df_avaliacoes_sem_id.count()} | "
    f"Avaliações: {resumo_avaliacoes['total']} | Duplicatas removidas: {duplicatas_removidas} | "
    f"Comentários padronizados: {resumo_avaliacoes['comentarios_padronizados']}"
)

In [ ]:
# tabela silver.tb_generos
# transforma listas com separadores diferentes em uma relação única de filme e gênero

df_generos_bronze = ler_bronze("generos")
validar_colunas(
    df_generos_bronze,
    {"id", "genres", "ingestion_datetime"},
    "bronze.tb_credits_and_tags",
)

# limita os valores ao domínio cinematográfico para eliminar resíduos de column shift
generos_validos = [
    "Action", "Adventure", "Animation", "Comedy", "Crime", "Documentary",
    "Drama", "Family", "Fantasy", "History", "Horror", "Music", "Mystery",
    "Romance", "Science Fiction", "TV Movie", "Thriller", "War", "Western",
]
mapa_generos = {genero.lower(): genero for genero in generos_validos}
mapa_generos_expr = F.create_map(
    *[item for chave, valor in mapa_generos.items() for item in (F.lit(chave), F.lit(valor))]
)

df_generos_base = (
    df_generos_bronze
    .withColumn("data_hora_ingestao", F.col("ingestion_datetime"))
    .withColumn("id_filme", F.trim(F.col("id").cast("string")))
    .withColumn("generos_texto", F.trim(F.coalesce(F.col("genres"), F.lit(""))))
    .where(F.col("id_filme").isNotNull() & (F.col("id_filme") != ""))
)
# seleciona a lista mais recente antes do explode para não misturar versões da mesma origem
df_generos_base = manter_ultima_carga_por_chave(df_generos_base, ["id_filme"])

# converte vírgula, ponto e vírgula e barra vertical em uma linha por gênero válido
df_generos = (
    df_generos_base
    .withColumn("generos_normalizados", F.regexp_replace("generos_texto", r"\s*[;|]\s*", ","))
    .withColumn("genero_bruto", F.explode(F.split("generos_normalizados", ",")))
    .withColumn("nome_genero", F.trim(F.regexp_replace("genero_bruto", r"[\[\]{}\"]", "")))
    .withColumn("nome_genero", mapa_generos_expr[F.lower("nome_genero")])
    .where(F.col("nome_genero").isNotNull())
)

df_generos_silver = (
    df_generos
    .select("id_filme", "nome_genero", "data_hora_ingestao")
    .dropDuplicates(["id_filme", "nome_genero"])
)
df_generos_validacao = salvar_silver(df_generos_silver, "generos")

In [ ]:
# validação de silver.tb_generos
# garante que cada par filme e gênero apareça uma vez e pertença ao domínio aceito

validar_coluna_ingestao_silver(df_generos_validacao, TABELAS["generos"]["destino"])
validar_tipos(
    df_generos_validacao,
    {"id_filme": "string", "nome_genero": "string", "data_hora_ingestao": "timestamp"},
    TABELAS["generos"]["destino"],
)

# confere pares repetidos, nomes vazios e valores fora do domínio
duplicados_generos = (
    df_generos_validacao.groupBy("id_filme", "nome_genero").count()
    .where(F.col("count") > 1).count()
)
resumo_generos = df_generos_validacao.agg(
    F.count("*").alias("total"),
    F.countDistinct("id_filme").alias("filmes"),
    contar_quando(F.col("id_filme").isNull() | (F.trim("id_filme") == "")).alias("ids_invalidos"),
    contar_quando(F.col("nome_genero").isNull() | (F.trim("nome_genero") == "")).alias("vazios"),
    contar_quando(~F.col("nome_genero").isin(generos_validos)).alias("fora_dominio"),
).first()

if duplicados_generos or resumo_generos["ids_invalidos"] or resumo_generos["vazios"] or resumo_generos["fora_dominio"]:
    raise AssertionError("Há duplicidades, valores vazios ou gêneros fora do domínio na Silver.")

display(df_generos_validacao.groupBy("nome_genero").count().orderBy(F.col("count").desc()))
print(f"Relações filme-gênero: {resumo_generos['total']} | Filmes: {resumo_generos['filmes']}")

In [ ]:
# tabela silver.tb_pessoas_empresas
# transforma quatro listas da origem em uma estrutura única de entidades por filme
# conserva o tipo de participação para separar pessoas e produtoras na gold

df_pessoas_bronze = ler_bronze("pessoas")
validar_colunas(
    df_pessoas_bronze,
    {"id", "cast", "directors", "writers", "production_companies", "ingestion_datetime"},
    "bronze.tb_credits_and_tags",
)

df_pessoas_base = (
    df_pessoas_bronze
    .withColumn("data_hora_ingestao", F.col("ingestion_datetime"))
    .withColumn("id_filme", F.trim(F.col("id").cast("string")))
    .withColumn("atores_texto", F.trim(F.coalesce(F.col("cast"), F.lit(""))))
    .withColumn("diretores_texto", F.trim(F.coalesce(F.col("directors"), F.lit(""))))
    .withColumn("roteiristas_texto", F.trim(F.coalesce(F.col("writers"), F.lit(""))))
    .withColumn("produtoras_texto", F.trim(F.coalesce(F.col("production_companies"), F.lit(""))))
    .where(F.col("id_filme").isNotNull() & (F.col("id_filme") != ""))
)
# seleciona a lista mais recente antes do explode para não combinar créditos de cargas diferentes
df_pessoas_base = manter_ultima_carga_por_chave(df_pessoas_base, ["id_filme"])

# empilha as quatro categorias e cria uma relação individual para cada entidade
df_pessoas = (
    df_pessoas_base
    .select(
        "id_filme", "data_hora_ingestao",
        F.expr(
            "stack(4, 'Ator', atores_texto, 'Diretor', diretores_texto, "
            "'Roteirista', roteiristas_texto, 'Produtora', produtoras_texto) "
            "AS (tipo_entidade, entidades_texto)"
        ),
    )
    .withColumn("entidade_bruta", F.explode(F.split("entidades_texto", r"\s*[,;|]\s*")))
    .withColumn("nome_entidade_base", F.trim(F.regexp_replace("entidade_bruta", r"[\[\]{}\"]", "")))
    .withColumn("nome_entidade_base", F.regexp_replace("nome_entidade_base", r"\s+", " "))
    .withColumn("motivo_rejeicao", classificar_residuo_entidade(F.col("nome_entidade_base")))
    .withColumn("nome_entidade", padronizar_nome_entidade(F.col("nome_entidade_base")))
)

# guarda os resíduos em um dataframe de auditoria antes de removê-los do resultado
df_entidades_rejeitadas = df_pessoas.where(F.col("motivo_rejeicao").isNotNull()).select(
    "id_filme", "tipo_entidade", "entidade_bruta", "motivo_rejeicao"
)
df_pessoas_silver = (
    df_pessoas
    .where(F.col("motivo_rejeicao").isNull())
    .select("id_filme", "nome_entidade", "tipo_entidade", "data_hora_ingestao")
    .dropDuplicates(["id_filme", "nome_entidade", "tipo_entidade"])
)
df_pessoas_validacao = salvar_silver(df_pessoas_silver, "pessoas")

In [ ]:
# validação de silver.tb_pessoas_empresas
# confirma tipos permitidos, nomes utilizáveis e ausência de relações repetidas

validar_coluna_ingestao_silver(df_pessoas_validacao, TABELAS["pessoas"]["destino"])
validar_tipos(
    df_pessoas_validacao,
    {
        "id_filme": "string", "nome_entidade": "string",
        "tipo_entidade": "string", "data_hora_ingestao": "timestamp",
    },
    TABELAS["pessoas"]["destino"],
)

# confere tipos permitidos, nomes válidos e relações não repetidas
duplicados_pessoas = (
    df_pessoas_validacao.groupBy("id_filme", "nome_entidade", "tipo_entidade").count()
    .where(F.col("count") > 1).count()
)
resumo_pessoas = df_pessoas_validacao.agg(
    F.count("*").alias("total"),
    F.countDistinct("id_filme").alias("filmes"),
    contar_quando(F.col("id_filme").isNull() | (F.trim("id_filme") == "")).alias("ids_invalidos"),
    contar_quando(~F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista", "Produtora")).alias("tipos_invalidos"),
    contar_quando(F.col("nome_entidade").isNull() | (F.trim("nome_entidade") == "")).alias("nomes_vazios"),
    contar_quando(F.col("nome_entidade") != F.trim(F.regexp_replace("nome_entidade", r"\s+", " "))).alias("espacos_irregulares"),
).first()
entidades_residuais = (
    df_pessoas_validacao
    .withColumn("motivo_rejeicao", classificar_residuo_entidade(F.col("nome_entidade")))
    .where(F.col("motivo_rejeicao").isNotNull())
    .count()
)

if any([
    duplicados_pessoas,
    resumo_pessoas["ids_invalidos"],
    resumo_pessoas["tipos_invalidos"],
    resumo_pessoas["nomes_vazios"],
    resumo_pessoas["espacos_irregulares"],
    entidades_residuais,
]):
    raise AssertionError("Há duplicidades, tipos inválidos ou resíduos em pessoas/empresas.")

display(df_entidades_rejeitadas.groupBy("motivo_rejeicao").count().orderBy(F.col("count").desc()))
display(
    df_entidades_rejeitadas
    .select("tipo_entidade", "entidade_bruta", "motivo_rejeicao")
    .dropDuplicates().orderBy("motivo_rejeicao", "entidade_bruta").limit(50)
)
display(df_pessoas_validacao.groupBy("tipo_entidade").count().orderBy(F.col("count").desc()))
print(
    f"Relações filme-entidade: {resumo_pessoas['total']} | Filmes: {resumo_pessoas['filmes']} | "
    f"Entidades rejeitadas: {df_entidades_rejeitadas.count()}"
)

In [ ]:
# tabela silver.tb_cotacao_dolar
# transforma os boletins da ptax em uma série diária contínua para a conversão financeira

df_cotacao_bronze = ler_bronze("cotacao")
validar_colunas(
    df_cotacao_bronze,
    {"dataHoraCotacao", "cotacaoCompra", "ingestion_datetime"},
    "bronze.tb_cotacao_dolar",
)

# recupera da bronze o período solicitado na landing para respeitar a parametrização original
periodo_solicitado = None
if {"data_inicio_periodo", "data_fim_periodo"}.issubset(df_cotacao_bronze.columns):
    periodo_solicitado = (
        df_cotacao_bronze
        .where(F.col("data_inicio_periodo").isNotNull() & F.col("data_fim_periodo").isNotNull())
        .orderBy(F.col("ingestion_datetime").desc())
        .select(
            F.to_date("data_inicio_periodo").alias("data_inicio"),
            F.to_date("data_fim_periodo").alias("data_fim"),
        )
        .first()
    )

# mantém o último boletim válido de cada dia antes de preencher as datas ausentes
df_cotacao_diaria = (
    df_cotacao_bronze
    .withColumn("data_hora_ingestao", F.col("ingestion_datetime"))
    .withColumn("data_hora_cotacao", F.to_timestamp("dataHoraCotacao"))
    .withColumn("data_cotacao", F.to_date("data_hora_cotacao"))
    .withColumn("cotacao_dolar_brl", F.col("cotacaoCompra").cast("DECIMAL(12,6)"))
    .where(F.col("data_cotacao").isNotNull() & (F.col("cotacao_dolar_brl") > 0))
)
janela_cotacao_dia = Window.partitionBy("data_cotacao").orderBy(
    F.col("data_hora_cotacao").desc(), F.col("data_hora_ingestao").desc()
)
df_cotacao_diaria = (
    df_cotacao_diaria
    .withColumn("_ordem_cotacao", F.row_number().over(janela_cotacao_dia))
    .where(F.col("_ordem_cotacao") == 1)
    .select("data_cotacao", "cotacao_dolar_brl", "data_hora_ingestao")
)

limites_observados = df_cotacao_diaria.agg(
    F.min("data_cotacao").alias("data_minima"),
    F.max("data_cotacao").alias("data_maxima"),
).first()
if limites_observados["data_minima"] is None:
    raise ValueError("A Bronze não possui cotações válidas.")

if periodo_solicitado is None:
    print("Aviso: limites dos widgets ausentes; usando o intervalo observado nas cotações.")
    data_inicio_periodo = limites_observados["data_minima"]
    data_fim_periodo = limites_observados["data_maxima"]
else:
    data_inicio_periodo = periodo_solicitado["data_inicio"]
    data_fim_periodo = periodo_solicitado["data_fim"]

if data_inicio_periodo > data_fim_periodo:
    raise ValueError("O período de cotação possui limites inválidos.")

# procura uma taxa-semente para preencher corretamente o primeiro fim de semana ou feriado
data_semente = (
    df_cotacao_diaria.where(F.col("data_cotacao") <= F.lit(data_inicio_periodo))
    .agg(F.max("data_cotacao").alias("data_semente"))
    .first()["data_semente"]
)
if data_semente is None:
    raise ValueError("Não há cotação-semente anterior ou igual ao início do período.")

# cria o calendário completo porque a api retorna somente os dias com boletim
# o preenchimento posterior mantém uma cotação disponível em fins de semana e feriados
df_calendario_cotacao = spark.range(1).select(
    F.explode(F.sequence(F.lit(data_semente), F.lit(data_fim_periodo))).alias("data_cotacao")
)
janela_forward_fill = Window.orderBy("data_cotacao").rowsBetween(
    Window.unboundedPreceding, Window.currentRow
)
# propaga a última cotação conhecida e depois mantém apenas o intervalo solicitado
df_cotacao_silver = (
    df_calendario_cotacao
    .join(df_cotacao_diaria, "data_cotacao", "left")
    .withColumn("data_origem_cotacao", F.when(F.col("cotacao_dolar_brl").isNotNull(), F.col("data_cotacao")))
    .withColumn("cotacao_dolar_brl", F.last("cotacao_dolar_brl", ignorenulls=True).over(janela_forward_fill))
    .withColumn("data_hora_ingestao", F.last("data_hora_ingestao", ignorenulls=True).over(janela_forward_fill))
    .withColumn("data_origem_cotacao", F.last("data_origem_cotacao", ignorenulls=True).over(janela_forward_fill))
    .withColumn("cotacao_preenchida", F.col("data_origem_cotacao") < F.col("data_cotacao"))
    .where(F.col("data_cotacao").between(F.lit(data_inicio_periodo), F.lit(data_fim_periodo)))
    .select(
        "data_cotacao", "data_origem_cotacao", "cotacao_preenchida",
        "cotacao_dolar_brl", "data_hora_ingestao",
    )
    .orderBy("data_cotacao")
)

df_cotacao_validacao = salvar_silver(df_cotacao_silver, "cotacao")

In [ ]:
# validação de silver.tb_cotacao_dolar
# garante cobertura diária completa e confirma a origem de cada valor preenchido

validar_coluna_ingestao_silver(df_cotacao_validacao, TABELAS["cotacao"]["destino"])
validar_tipos(
    df_cotacao_validacao,
    {
        "data_cotacao": "date", "data_origem_cotacao": "date",
        "cotacao_preenchida": "boolean", "cotacao_dolar_brl": "decimal(12,6)",
        "data_hora_ingestao": "timestamp",
    },
    TABELAS["cotacao"]["destino"],
)

# confere continuidade, origem das taxas e ausência de datas repetidas
duplicados_cotacao = (
    df_cotacao_validacao.groupBy("data_cotacao").count()
    .where(F.col("count") > 1).count()
)
resumo_cotacao = df_cotacao_validacao.agg(
    F.min("data_cotacao").alias("data_minima"),
    F.max("data_cotacao").alias("data_maxima"),
    F.count("data_cotacao").alias("quantidade_datas"),
    contar_quando(F.col("cotacao_dolar_brl").isNull()).alias("cotacoes_vazias"),
    contar_quando(
        F.col("data_origem_cotacao").isNull()
        | (F.col("data_origem_cotacao") > F.col("data_cotacao"))
    ).alias("origens_invalidas"),
    contar_quando(
        F.col("cotacao_preenchida") != (F.col("data_origem_cotacao") < F.col("data_cotacao"))
    ).alias("marcadores_incorretos"),
    contar_quando(F.col("cotacao_preenchida")).alias("datas_preenchidas"),
).first()
primeira_cotacao = df_cotacao_validacao.orderBy("data_cotacao").first()
quantidade_datas_esperada = (data_fim_periodo - data_inicio_periodo).days + 1

if duplicados_cotacao:
    raise AssertionError("Há datas de cotação duplicadas na Silver.")
if resumo_cotacao["data_minima"] != data_inicio_periodo or resumo_cotacao["data_maxima"] != data_fim_periodo:
    raise AssertionError("A Silver não cobre exatamente o período informado.")
if resumo_cotacao["quantidade_datas"] != quantidade_datas_esperada:
    raise AssertionError("O calendário de cotações possui datas faltantes.")
if any(resumo_cotacao[c] for c in ["cotacoes_vazias", "origens_invalidas", "marcadores_incorretos"]):
    raise AssertionError("O forward fill da cotação possui valores ou marcadores inválidos.")
if primeira_cotacao["data_origem_cotacao"] != data_semente:
    raise AssertionError("A primeira data não utiliza a cotação-semente esperada.")

display(df_cotacao_validacao.where(F.col("cotacao_preenchida")).orderBy("data_cotacao"))
print(
    f"Período: {data_inicio_periodo} a {data_fim_periodo} | Datas: {resumo_cotacao['quantidade_datas']} | "
    f"Preenchidas por forward fill: {resumo_cotacao['datas_preenchidas']}"
)

In [ ]:
# resumo da camada silver
# apresenta as sete saídas para facilitar a conferência do workflow antes da gold

# relê todas as saídas para apresentar o volume final de cada tabela
resumo_tabelas = [
    (TABELAS[chave]["destino"], ler_silver(chave).count())
    for chave in ["filmes", "financeiro", "metricas", "avaliacoes", "generos", "pessoas", "cotacao"]
]
display(spark.createDataFrame(resumo_tabelas, ["tabela", "quantidade_registros"]))